In [1]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
df = pd.read_csv("/content/drive/MyDrive/HealthTrack/healthtrack_raw.csv")

# Create clean dataset
df_clean = df.dropna(subset=["patient_id", "readmission_30d"])
df_clean = df_clean[df_clean["total_bill_usd"] != -9999]
print(df_clean.shape)  # Should be (230, 12)

Mounted at /content/drive
(230, 12)


In [2]:
# 9.One-hot encode categorical columns
df_encoded = pd.get_dummies(df_clean, columns=["department", "discharge_disposition", "insurance_type"])

# 10.Features and target
X = df_encoded.drop(columns=["patient_id", "admit_date", "discharge_date", "readmission_30d"])
y = df_encoded["readmission_30d"]

print("Features shape:", X.shape)
print("Target shape:", y.shape)

Features shape: (230, 19)
Target shape: (230,)


In [3]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier

# 11.split 80/20
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 12.Train model
model = GradientBoostingClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

print("Training done!")
print(f"Train size: {X_train.shape[0]} rows")
print(f"Test size: {X_test.shape[0]} rows")

Training done!
Train size: 184 rows
Test size: 46 rows


In [4]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# 13.Evaluation
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print(f"Accuracy:  {accuracy_score(y_test, y_pred):.3f}")
print(f"Precision: {precision_score(y_test, y_pred):.3f}")
print(f"Recall:    {recall_score(y_test, y_pred):.3f}")
print(f"F1 Score:  {f1_score(y_test, y_pred):.3f}")
print(f"ROC-AUC:   {roc_auc_score(y_test, y_prob):.3f}")

Accuracy:  0.783
Precision: 0.643
Recall:    0.643
F1 Score:  0.643
ROC-AUC:   0.754


Key insight:

Recall (0.643) is the most important metric here — a missed high-risk patient (false negative) means a preventable readmission costing $15,200+ in penalties. A false positive just means an unnecessary follow-up call, which is far less harmful.

The recall of 0.643 is modest but expected given the small dataset
(230 rows). In production, this model would be retrained on full
EHR data (10,000+ records) with richer features like diagnosis codes
and lab results, which would significantly improve recall. For now,
it serves as a proof-of-concept pipeline.

In [5]:
# 14.Add probability and risk tier columns
df_clean = df_clean.copy()
df_clean["readmission_probability"] = model.predict_proba(
    df_encoded.drop(columns=["patient_id", "admit_date", "discharge_date", "readmission_30d"])
)[:, 1]

df_clean["risk_tier"] = df_clean["readmission_probability"].apply(
    lambda x: "High" if x >= 0.65 else ("Medium" if x >= 0.40 else "Low")
)

print(df_clean["risk_tier"].value_counts())

risk_tier
Low       138
High       87
Medium      5
Name: count, dtype: int64


In [6]:
df_clean.to_csv("/content/drive/MyDrive/HealthTrack/healthtrack_clean.csv", index=False)
print("Saved!")
print(df_clean[["patient_id", "readmission_probability", "risk_tier"]].head(10))

Saved!
   patient_id  readmission_probability risk_tier
0       P0001                 0.024175       Low
1       P0002                 0.041451       Low
2       P0003                 0.231725       Low
3       P0004                 0.970258      High
29      P0030                 0.865961      High
30      P0031                 0.009224       Low
31      P0032                 0.984218      High
32      P0033                 0.019035       Low
33      P0034                 0.081375       Low
34      P0035                 0.290688       Low


###Part D — LangChain Alert Generation!

In [10]:
!pip install langchain langchain-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.8/119.8 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.9/554.9 kB 11.9 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.4.3
    Uninstalling langchain-core-1.4.3:
      Successfully uninstalled langchain-core-1.4.3


In [12]:
!pip install langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 3.2 MB/s eta 0:00:00


In [16]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from google.colab import userdata

GROQ_API_KEY = userdata.get('GROQ_API_KEY')

llm = ChatGroq(
    model="llama-3.1-8b-instant",  # updated model
    api_key=GROQ_API_KEY,
    temperature=0.3
)


# 15,16.Prompt template
prompt = PromptTemplate(
    input_variables=["patient_id", "age", "department", "length_of_stay",
                     "prev_admissions_12m", "num_medications", "discharge_disposition"],
    template="""
You are a clinical care coordinator. Write a 3-sentence clinical alert note for this patient.
Sentence 1: State the risk tier and top 2 contributing factors.
Sentence 2: Recommend a specific action.
Sentence 3: Professional closing with urgency level.

Patient ID: {patient_id}
Age: {age}
Department: {department}
Length of Stay: {length_of_stay} days
Previous Admissions (12m): {prev_admissions_12m}
Number of Medications: {num_medications}
Discharge Disposition: {discharge_disposition}

Write only the 3 sentences, no headings or bullet points.
"""
)

print("Setup done!")


Setup done!


In [17]:
# 17.Filter only High risk patients
high_risk = df_clean[df_clean["risk_tier"] == "High"].copy()
print(f"Generating alerts for {len(high_risk)} High risk patients...")

# Generate notes
intervention_notes = []

for _, row in high_risk.iterrows():
    formatted_prompt = prompt.format(
        patient_id=row["patient_id"],
        age=row["age"],
        department=row["department"],
        length_of_stay=row["length_of_stay"],
        prev_admissions_12m=row["prev_admissions_12m"],
        num_medications=row["num_medications"],
        discharge_disposition=row["discharge_disposition"]
    )
    response = llm.invoke(formatted_prompt)
    intervention_notes.append(response.content)

high_risk["intervention_note"] = intervention_notes

# Add alert priority
high_risk["alert_priority"] = high_risk.apply(
    lambda row: "URGENT" if row["prev_admissions_12m"] >= 2 else "STANDARD", axis=1
)

print("Done!")
print(high_risk[["patient_id", "risk_tier", "alert_priority", "intervention_note"]].head(3))

Generating alerts for 87 High risk patients...
Done!
   patient_id risk_tier alert_priority  \
3       P0004      High       STANDARD   
29      P0030      High         URGENT   
31      P0032      High       STANDARD   

                                    intervention_note  
3   Patient P0004 is classified as high risk due t...  
29  Patient P0030 is at high risk for readmission ...  
31  Patient P0032 is classified as high-risk due t...  


In [18]:
# Select final 7 columns
alerts_df = high_risk[[
    "patient_id", "age", "department",
    "risk_tier", "readmission_probability",
    "intervention_note", "alert_priority"
]]

alerts_df.to_csv("/content/drive/MyDrive/HealthTrack/healthtrack_alerts_today.csv", index=False)
print(f"Saved {len(alerts_df)} alerts!")
print(alerts_df.head())

Saved 87 alerts!
   patient_id  age        department risk_tier  readmission_probability  \
3       P0004   37          Oncology      High                 0.970258   
29      P0030   92  General Medicine      High                 0.865961   
31      P0032   76  General Medicine      High                 0.984218   
70      P0071   90        Cardiology      High                 0.992452   
72      P0073   84       Orthopedics      High                 0.990971   

                                    intervention_note alert_priority  
3   Patient P0004 is classified as high risk due t...       STANDARD  
29  Patient P0030 is at high risk for readmission ...         URGENT  
31  Patient P0032 is classified as high-risk due t...       STANDARD  
70  Patient P0071 is at high risk for readmission,...         URGENT  
72  Patient P0073 is at high risk for readmission ...         URGENT  


## Part D — LangChain Alert Generation

- Model: llama-3.1-8b-instant via Groq API
- Alerts generated for 87 High risk patients only
- Each note contains: risk tier, top 2 factors, recommended action
- URGENT: prev_admissions_12m >= 2 AND risk_tier == High
- STANDARD: all other High risk patients